## Setup

Import the libraries and define the source and output dataset locations.

In [ ]:
import random
import os
import numpy as np
import matplotlib.pyplot as plt
import nibabel as nib
from collections import Counter
from tqdm import tqdm
import shutil

In [ ]:
data_path = r"/kaggle/neurite_oasis"

In [ ]:
output_data_dir = r"/kaggle/neurite-oasis-split"
test_dir = os.path.join(output_data_dir, "test")
val_dir = os.path.join(output_data_dir, "val")

print(f"Output data directory: {output_data_dir}")
print(f"Test directory: {test_dir}")
print(f"Val directory: {val_dir}")

## Dataset Overview

Count the available subjects and inspect a few examples before building the split.

In [ ]:
# Calculate total number of subjects
# the folder name for each subject in the pattern: OASIS_OAS1_xxxx_MR1

subject_folders = [f for f in os.listdir(data_path) if f.startswith("OASIS_OAS1_") and f.endswith("_MR1")]
total_subjects = len(subject_folders)
print(f"Total number of subjects: {total_subjects}")


In [ ]:
sample_subjects = random.sample(subject_folders, min(10, len(subject_folders)))
print(f"Visualizing {len(sample_subjects)} random subjects:")
for subject in sample_subjects:
    print(f"  {subject}")

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes = axes.flatten()

for idx, subject in enumerate(sample_subjects):
    norm_file = os.path.join(data_path, subject, "aligned_norm.nii.gz")
    if os.path.exists(norm_file):
        img = nib.load(norm_file)
        data = img.get_fdata()
        
        middle_slice = data.shape[2] // 2
        axes[idx].imshow(data[:, :, middle_slice], cmap='gray')
        axes[idx].set_title(subject[:20], fontsize=8)
        axes[idx].axis('off')
    else:
        axes[idx].text(0.5, 0.5, 'File not found', ha='center', va='center')
        axes[idx].set_title(subject[:20], fontsize=8)
        axes[idx].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
dimension_voxel_info = []

for subject in tqdm(subject_folders, desc="Processing subjects"):
    norm_file = os.path.join(data_path, subject, "aligned_norm.nii.gz")
    if os.path.exists(norm_file):
        img = nib.load(norm_file)
        data = img.get_fdata()
        affine = img.affine
        
        dimension = tuple(data.shape)
        voxel_spacing = tuple(np.sqrt(np.sum(affine[:3, :3] ** 2, axis=0)))
        
        dimension_voxel_info.append({
            'subject': subject,
            'dimension': dimension,
            'voxel_spacing': voxel_spacing
        })

print(f"Total subjects processed: {len(dimension_voxel_info)}")

In [ ]:
dimension_voxel_combinations = [(info['dimension'], tuple(np.round(info['voxel_spacing'], 4))) for info in dimension_voxel_info]
combination_counter = Counter(dimension_voxel_combinations)

print(f"\nTotal unique dimension/voxel_spacing combinations: {len(combination_counter)}\n")
print("Top 5 most unique combinations:")
print("=" * 80)

for rank, (combo, count) in enumerate(combination_counter.most_common(5), 1):
    dimension, voxel_spacing = combo
    print(f"{rank}. Dimension: {dimension} | Voxel Spacing: {voxel_spacing}")
    print(f"   Count: {count} subjects")
    print()

## Split Creation

Create disjoint validation and test subsets with 50 random patients each, then copy the required files.

In [ ]:
os.makedirs(test_dir, exist_ok=True)
os.makedirs(val_dir, exist_ok=True)

num_subjects = len(subject_folders)
subjects_per_split = 50

if num_subjects < subjects_per_split * 2:
    raise ValueError(f"Need at least {subjects_per_split * 2} subjects to create val/test splits of {subjects_per_split} each. Found {num_subjects}.")

shuffled_subjects = random.sample(subject_folders, num_subjects)
val_subjects = shuffled_subjects[:subjects_per_split]
test_subjects = shuffled_subjects[subjects_per_split:subjects_per_split * 2]

print(f"Total subjects: {num_subjects}")
print(f"Val subjects: {len(val_subjects)}")
print(f"Test subjects: {len(test_subjects)}")

In [ ]:
for subject in tqdm(val_subjects, desc="Copying val subjects"):
    dest_dir = os.path.join(val_dir, subject)
    os.makedirs(dest_dir, exist_ok=True)

    for fname in ("aligned_norm.nii.gz", "aligned_seg35.nii.gz"):
        src_file = os.path.join(data_path, subject, fname)
        dest_file = os.path.join(dest_dir, fname)
        if os.path.exists(src_file):
            shutil.copy2(src_file, dest_file)
        else:
            print(f"Warning: {src_file} not found")

print(f"Val split complete: {len(val_subjects)} subjects copied")

In [ ]:
for subject in tqdm(test_subjects, desc="Copying test subjects"):
    dest_dir = os.path.join(test_dir, subject)
    os.makedirs(dest_dir, exist_ok=True)

    for fname in ("aligned_norm.nii.gz", "aligned_seg35.nii.gz"):
        src_file = os.path.join(data_path, subject, fname)
        dest_file = os.path.join(dest_dir, fname)
        if os.path.exists(src_file):
            shutil.copy2(src_file, dest_file)
        else:
            print(f"Warning: {src_file} not found")

print(f"Test split complete: {len(test_subjects)} subjects copied")

## Segmentation Check

Run a quick sanity check on a few segmentation masks to confirm dimensions and label counts.

In [ ]:
# Analyze segmentation mask dimensions for 5 subjects
sample_size = 5
sample_subjects_seg = random.sample(subject_folders, min(sample_size, len(subject_folders)))
seg_dimension_info = []

for subject in tqdm(sample_subjects_seg, desc="Processing segmentation masks"):
    seg_file = os.path.join(data_path, subject, "aligned_seg35.nii.gz")
    if os.path.exists(seg_file):
        img = nib.load(seg_file)
        data = img.get_fdata()

        dimension = tuple(data.shape)
        num_labels = len(np.unique(data))

        seg_dimension_info.append({
            'subject': subject,
            'seg_dimension': dimension,
            'num_labels': num_labels
        })
    else:
        print(f"Warning: aligned_seg35.nii.gz not found for {subject}")

print(f"Total segmentation masks processed: {len(seg_dimension_info)}")

print(f"\nSegmentation dimensions:")
for info in seg_dimension_info:
    print(f"  {info['subject']}: {info['seg_dimension']} | Labels: {info['num_labels']}")